In [7]:
# Imports & Setup
import sys
import pandas as pd
import matplotlib.pyplot as plt

# Ensure local modules under src/ can be imported
sys.path.append("src")
from project1 import get_coordinates, fetch_weather_forecast
from utils import get_rng  # Reusable RNG setup from earlier

# Seed generator if adding random analysis or noise modeling
rng = get_rng(seed=42)

ModuleNotFoundError: No module named 'matplotlib'

In [9]:
# Fetch and Process Data
city_query = "London"
lat, lon, display_name = get_coordinates(city_query)
forecast_raw = fetch_weather_forecast(lat, lon, days=7)

# Extract daily dataset into a clean Pandas DataFrame
daily_data = forecast_raw["daily"]
df_forecast = pd.DataFrame({
    "Date": daily_data["time"],
    "Max Temp (°C)": daily_data["temperature_2m_max"],
    "Min Temp (°C)": daily_data["temperature_2m_min"],
    "Precipitation (mm)": daily_data["precipitation_sum"],
    "Max Wind (km/h)": daily_data["wind_speed_10m_max"]
})

print(f"Weather Forecast for: {display_name}")
df_forecast

NameError: name 'get_coordinates' is not defined

In [5]:
# Display / Plot Findings
plt.figure(figsize=(10, 5))
plt.plot(df_forecast["Date"], df_forecast["Max Temp (°C)"], label="Max Temp (°C)", marker="o")
plt.plot(df_forecast["Date"], df_forecast["Min Temp (°C)"], label="Min Temp (°C)", marker="o")
plt.title(f"7-Day Temperature Forecast for {display_name}")
plt.xlabel("Date")
plt.ylabel("Temperature (°C)")
plt.grid(True)
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

NameError: name 'plt' is not defined

In [ ]:
# Part B: Weather Forecast Program

This section uses reusable logic from `src/project1.py` to:
1. Accept a target location name.
2. Resolve coordinates using the Open-Meteo Geocoding API.
3. Fetch daily weather forecast data (temperatures, precipitation, wind speed).
4. Display the forecast clearly as a Pandas DataFrame and plot multi-day trends.

In [ ]:
import sys
import pandas as pd
import matplotlib.pyplot as plt

# Import reusable logic from src/ project structure
sys.path.append("src")
from project1 import get_coordinates, fetch_weather_forecast
from utils import get_rng

# 1. Take location name and resolve coordinates
location_query = "Dresden"
lat, lon, display_name = get_coordinates(location_query)

print(f"Location Resolved: {display_name}")
print(f"Coordinates: Latitude {lat:.4f}, Longitude {lon:.4f}\n")

# 2. Fetch multi-day forecast via Open-Meteo API
raw_forecast = fetch_weather_forecast(lat, lon, days=7)

# 3. Format into a clear Pandas DataFrame
daily_data = raw_forecast["daily"]
df_weather = pd.DataFrame({
    "Date": daily_data["time"],
    "Max Temp (°C)": daily_data["temperature_2m_max"],
    "Min Temp (°C)": daily_data["temperature_2m_min"],
    "Precipitation (mm)": daily_data["precipitation_sum"],
    "Max Wind Speed (km/h)": daily_data["wind_speed_10m_max"]
})

# Display formatted table
print("7-Day Weather Forecast:")
df_weather

In [ ]:
# Display clear, formatted forecast plots
plt.figure(figsize=(10, 5))

# Plot temperatures
plt.plot(df_weather["Date"], df_weather["Max Temp (°C)"], marker="o", color="crimson", linewidth=2, label="Max Temp (°C)")
plt.plot(df_weather["Date"], df_weather["Min Temp (°C)"], marker="o", color="dodgerblue", linewidth=2, label="Min Temp (°C)")

plt.title(f"7-Day Weather Forecast for {display_name}", fontsize=14, fontweight="bold")
plt.xlabel("Date", fontsize=11)
plt.ylabel("Temperature (°C)", fontsize=11)
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend()
plt.xticks(rotation=30)
plt.tight_layout()

plt.show()

In [ ]:
### Summary of Weather Findings
* **Location Resolution**: Successfully converted location query into precise coordinates.
* **Forecast Highlights**: Tracked expected max/min daily temperature trends, precipitation probability, and wind speeds over a 7-day period.
* **API Reliability**: Standardized query formatting via `requests` ensures clean response parsing and graceful error handling.

In [ ]:
# Part C: Multi-Source Forecast Comparison (Open-Meteo vs MET Norway)

lat, lon, display_name = get_coordinates("Dresden")

# Source 1: Open-Meteo
open_meteo_raw = fetch_open_meteo_forecast(lat, lon, days=1)
open_max_temp = open_meteo_raw["daily"]["temperature_2m_max"][0]

# Source 2: MET Norway
met_raw = fetch_met_norway_forecast(lat, lon)
# Extract instant 2m air temperature from closest forecast time block
met_instant_temp = met_raw["properties"]["timeseries"][0]["data"]["instant"]["details"]["air_temperature"]

print(f"Weather Comparison for {display_name}:")
print(f" - Open-Meteo (Max Temp Today): {open_max_temp}°C")
print(f" - MET Norway (Current Instant Temp): {met_instant_temp}°C")

# Explicit Citation Requirement
print("\nData Sources Cites:")
print(" - Open-Meteo API (https://open-meteo.com)")
print(" - MET Norway Locationforecast 2.0 API (https://api.met.no)")

In [ ]:
# Part A: HTTP Fundamentals (urllib vs requests)

In this section, we compare querying web APIs using Python's standard `urllib` library versus the third-party `requests` library by geocoding a city name ("Dresden").

In [ ]:
import json
import time
import urllib.parse
import urllib.request

# 1. Geocode "Dresden" using urllib
base_url = "https://geocoding-api.open-meteo.com/v1/search"
query_params = {
    "name": "Dresden",
    "count": 1,
    "language": "en",
    "format": "json"
}

# Build query string cleanly with urllib.parse.urlencode
encoded_params = urllib.parse.urlencode(query_params)
full_url = f"{base_url}?{encoded_params}"

# Custom User-Agent as per project requirements
headers = {
    "User-Agent": "UCU-Student-WeatherApp/1.0 (b41134@students.ucu.ac)"
}

start_time = time.time()
req = urllib.request.Request(full_url, headers=headers)

with urllib.request.urlopen(req, timeout=10) as response:
    urllib_status = response.getcode()
    urllib_content_type = response.headers.get("Content-Type")
    raw_body = response.read().decode("utf-8")
    urllib_data = json.loads(raw_body)

urllib_elapsed = time.time() - start_time

print("--- urllib.request Execution ---")
print(f"URL: {full_url}")
print(f"Status Code: {urllib_status}")
print(f"Content-Type: {urllib_content_type}")
print(f"Response Time: {urllib_elapsed:.4f} seconds")
print(f"Geocoded Result: {urllib_data['results'][0]['name']}, {urllib_data['results'][0]['country']} (Lat: {urllib_data['results'][0]['latitude']}, Lon: {urllib_data['results'][0]['longitude']})")

In [ ]:
import requests

# 2. Geocode "Dresden" using requests with params and timeout
requests_headers = {
    "User-Agent": "UCU-Student-WeatherApp/1.0 (b41134@students.ucu.ac)"
}

start_time = time.time()
response = requests.get(base_url, params=query_params, headers=requests_headers, timeout=10)
requests_elapsed = time.time() - start_time

# 4. Inspect and report status code, Content-Type, and response time
print("--- requests Execution ---")
print(f"Final URL: {response.url}")
print(f"Status Code: {response.status_code}")
print(f"Content-Type Header: {response.headers.get('Content-Type')}")
print(f"Response Time (Server Latency): {response.elapsed.total_seconds():.4f} seconds")
print(f"Total Roundtrip Time: {requests_elapsed:.4f} seconds")

requests_data = response.json()
print(f"Geocoded Result: {requests_data['results'][0]['name']}, {requests_data['results'][0]['country']} (Lat: {requests_data['results'][0]['latitude']}, Lon: {requests_data['results'][0]['longitude']})")

In [ ]:
### Comparison of Approaches: `urllib` vs `requests`

| Feature / Metric | Standard Library (`urllib`) | Third-Party Library (`requests`) |
| :--- | :--- | :--- |
| **Code Length & Verbosity** | Requires multiple imports (`urllib.request`, `urllib.parse`, `json`). Parameter encoding (`urlencode`), byte decoding (`.decode('utf-8')`), and JSON parsing (`json.loads()`) must all be executed manually. | Extremely concise. Pass parameters directly using `params= query_params` and parse JSON in one step via `response.json()`. |
| **Error Handling** | Raises low-level exceptions like `urllib.error.HTTPError` or `urllib.error.URLError`. Checking for HTTP failures requires explicit `try...except` blocks or manual status code verification. | Simple error handling via `response.raise_for_status()`, which automatically raises HTTP exceptions for 4xx/5xx responses. |
| **Header & Encoding Control** | Headers require instantiating a separate `urllib.request.Request` object before opening the connection. Character encodings must be manually parsed from response headers or assumed (`utf-8`). | Headers are passed directly as a dictionary argument (`headers=...`). The library automatically detects encoding from response headers and decodes content automatically (`response.text`). |
| **Timeout Handling** | Specified as a numeric argument (`timeout=10`) in `urlopen()`. | Specified directly as a keyword argument (`timeout=10`) in `requests.get()`. |

In [ ]:
import sys
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append("src")
from project1 import geocode, get_forecast, WMO_WEATHER_CODES

# 1. Candidate lookup
city_name = "Springfield"
lat, lon, display_name, tz = geocode(city_name, selection_index=0)

print(f"Selected Location: {display_name}")
print(f"Coordinates: ({lat:.4f}, {lon:.4f}) | Timezone: {tz}\n")

# 2. Fetch forecast matching location timezone
forecast_raw = get_forecast(lat, lon, days=7, timezone=tz)
daily_data = forecast_raw["daily"]

# 3. Convert into pandas DataFrame with clear column names, units, & decoded weather codes
df_forecast = pd.DataFrame({
    "Date": daily_data["time"],
    "Max Temp (°C)": daily_data["temperature_2m_max"],
    "Min Temp (°C)": daily_data["temperature_2m_min"],
    "Precipitation (mm)": daily_data["precipitation_sum"],
    "Max Wind Speed (km/h)": daily_data["wind_speed_10m_max"],
    "Weather Code": daily_data["weather_code"],
    "Condition": [WMO_WEATHER_CODES.get(code, f"Unknown ({code})") for code in daily_data["weather_code"]]
})

# Display Formatted Table
print("--- 7-Day Weather Forecast Table ---")
df_forecast

In [ ]:
# 4. Chart of Temperature Range with Precipitation
fig, ax1 = plt.subplots(figsize=(10, 5))

# Plot Temperature Range (Max and Min) on primary Y axis
color_max = "#d9534f"
color_min = "#0275d8"
ax1.plot(df_forecast["Date"], df_forecast["Max Temp (°C)"], marker="o", color=color_max, linewidth=2, label="Max Temp (°C)")
ax1.plot(df_forecast["Date"], df_forecast["Min Temp (°C)"], marker="o", color=color_min, linewidth=2, label="Min Temp (°C)")
ax1.set_xlabel("Date", fontsize=11, fontweight="bold")
ax1.set_ylabel("Temperature (°C)", fontsize=11, fontweight="bold")
ax1.tick_params(axis="x", rotation=30)
ax1.grid(True, linestyle="--", alpha=0.5)

# Plot Precipitation on secondary Y axis
ax2 = ax1.twinx()
color_precip = "#5bc0de"
ax2.bar(df_forecast["Date"], df_forecast["Precipitation (mm)"], alpha=0.4, color=color_precip, width=0.4, label="Precipitation (mm)")
ax2.set_ylabel("Precipitation (mm)", fontsize=11, fontweight="bold", color="#1b6ec2")

# Combine legends from both axes
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

plt.title(f"7-Day Weather Forecast & Precipitation Range: {display_name}", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
### Weather Forecast Summary
* **Location & Timezone**: Forecast generated for **Springfield** matching timezone settings.
* **Temperature Dynamics**: Expected daily peak temperature reaches **{max_temp}°C** with night lows dropping to **{min_temp}°C**.
* **Precipitation Outlook**: Total precipitation accumulates to **{total_precip:.1f} mm** over the 7-day period.
* **Prevalent Condition**: Most frequent weather condition decoded as **{main_condition}**.

In [3]:
%matplotlib inline
import sys
import pandas as pd
import matplotlib.pyplot as plt

# Import custom modules
sys.path.append("src")
from project1 import geocode, get_forecast, WMO_WEATHER_CODES

Matplotlib is building the font cache; this may take a moment.


In [ ]:
# Part C: Multi-Source Comparison (Open-Meteo vs MET Norway)

from project1 import get_met_norway_forecast

# 1. Fetch MET Norway forecast for the same coordinates
met_data = get_met_norway_forecast(lat, lon)

# Extract instant temperature from the latest MET Norway timeseries block
current_met_temp = met_data["properties"]["timeseries"][0]["data"]["instant"]["details"]["air_temperature"]
open_meteo_max_today = df_forecast["Max Temp (°C)"].iloc[0]

print("--- Multi-Source Weather Comparison ---")
print(f"Location: {display_name}")
print(f"Open-Meteo (Today's Max Temp): {open_meteo_max_today}°C")
print(f"MET Norway (Current Instant Temp): {current_met_temp}°C")

# Required Data Citation Output
print("\nData Source Citations:")
print(" - Open-Meteo Weather API: https://open-meteo.com/ (CC BY 4.0)")
print(" - MET Norway Locationforecast 2.0: https://api.met.no/ (NLOD / CC BY 4.0)")

In [ ]:
# Part C2: Robustness Demonstrations with Broken Inputs

import pytest
from src.project1 import geocode, get_forecast, safe_request, APIError

print("=== Demonstration 1: Invalid City Name ===")
try:
    geocode("NonExistentCityXYZ12389")
except ValueError as e:
    print(f"Handled Expected Exception -> {e}\n")

print("=== Demonstration 2: Invalid Latitude ===")
try:
    # Latitude out of valid range (-90 to 90)
    get_forecast(lat=999.0, lon=0.0)
except APIError as e:
    print(f"Handled Expected Exception -> {e}\n")

print("=== Demonstration 3: Tiny Timeout ===")
try:
    # 0.001 second timeout to force a Timeout error
    safe_request("https://api.open-meteo.com/v1/forecast", params={"latitude": 0, "longitude": 0}, timeout=0.001)
except APIError as e:
    print(f"Handled Expected Exception -> {e}\n")

print("=== Demonstration 4: Simulated Network Failure ===")
try:
    # Requesting an invalid non-existent domain to simulate no network
    safe_request("https://this-domain-does-not-exist-123456.com", max_retries=2, backoff_factor=0.01)
except APIError as e:
    print(f"Handled Expected Exception -> {e}\n")

In [ ]:
Part D: Technical & Architecture Report1. 
    1. System Architecture & Data Flow
The weather forecast application is designed as a modular Python pipeline that transforms
user-provided location queries into formatted daily weather forecasts, cross-source 
comparisons, and dual-axis visual charts. The end-to-end data processing lifecycle consists
of four primary stages: User Input & Geocoding, Forecast Fetching,
Data Transformation & Normalization, and Presentation.[ User Input: "Springfield" ]
             │
             ▼
[ 1. Geocoding API (Open-Meteo) ]
             │  (Lat, Lon, Timezone, Candidates)
             ▼
[ Candidate Selection / Validation ]
             │
      ┌──────┴────────────────────────┐
      ▼                               ▼
[ 2a. Primary API (Open-Meteo) ]   [ 2b. Secondary API (MET Norway) ]
      │ (7-Day Daily Forecast)        │ (Instant Location Forecast)
      └──────┬────────────────────────┘
             ▼
[ 3. Pandas Transformation & Decoding ]
             │  (WMO Weather Codes -> Descriptions, Units)
             ▼
[ 4. Presentation Layer ]
      ├── Formatted Pandas DataFrame Table
      ├── Dual-Axis Matplotlib Visualization
      └── Natural Language Summary Report

Step-by-Step Data LifecycleUser 
1. Input & Geocoding Query:
The user inputs a location string (e.g., "Springfield"). The system dispatches an 
    HTTP GET request to Open-Meteo's Geocoding API endpoint 
        ([https://geocoding-api.open-meteo.com/v1/search](https://geocoding-api.open-meteo.com/v1/search)). 
The response returns candidate locations containing essential spatial metadata: 
    latitude, longitude, administrative region, country, and the local IANA timezone 
identifier (e.g., America/Chicago). If multiple candidates match, selection indices 
allow explicit spatial targeting.
    2. Parallel Forecast Retrieval:
Using the validated spatial coordinates (latitude, longitude) and local timezone, 
two asynchronous HTTP requests are sent to fetch forecast data:
    Primary Source (Open-Meteo): Queries daily maximum and minimum temperatures
, total precipitation sum, maximum wind speed, and WMO weather codes over a 7-day
forecast horizon. Matching the local timezone ensures daily aggregation windows 
align accurately with local midnight-to-midnight hours rather than UTC.
    Secondary Source (MET Norway): Queries the locationforecast/2.0/compact endpoint
using coordinate values rounded to four decimal places. 
This delivers location-based point forecasts used for verification and cross-source 
    comparison.
    Data Transformation & Dataframe Structuring:The raw JSON response structures returned
by the APIs are extracted into a structured pandas.DataFrame. 
Raw WMO weather code integers (e.g., 0, 61, 95) are dynamically decoded into natural 
language descriptions (e.g., "Clear sky", "Slight rain", "Thunderstorm") using an 
internal lookup dictionary mapped from official Open-Meteo meteorological definitions.
    Presentation & Plotting:The formatted DataFrame renders an interactive tabular 
view in the notebook environment. Concurrently, matplotlib renders a dual-axis 
visualization: primary $Y$-axis plotting maximum and minimum temperature trend lines, 
and a secondary $Y$-axis displaying precipitation volume as a bar plot.
    
    2. Licensing, Terms of Service, and Commercial Realities
Building reliable software products requires strict adherence to API licensing, attribution rules, and rate limit 
constraints. The table below compares the legal and operational frameworks of our two 
data providers:
